In [1]:
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import math
import pandas as pd

In [2]:
# Todo: Improve chunking for larger datasets
## from dask.distributed import Client, LocalCluster
## client = Client()
## client

In [ ]:
bios_output_path = "/g/data/rp23/experiments/2024-04-17_BIOS3-merge/BIOS3_output/"
trendy_output_path = "/g/data/rp23/experiments/2024-04-17_BIOS3-merge/lw5085/BIOS_through_TRENDY"

landmask_path = "/g/data/rp23/experiments/2024-04-17_BIOS3-merge/lw5085/landmasks"

bios_act9_output = "ACT9/S2_mpi"
trendy_act9_output = "S2-act9/output"

In [ ]:
veg_type_map = {
    -2 : "peb",
    2  : "seb",
    6  : "c3",
    7  : "c4",
}

In [5]:
test_landmask = f"{landmask_path}/Australia_BIOS_9pts_at_0p05_resolution_landmask.nc"
landmask = xr.open_dataset(test_landmask)

In [6]:
bios_cable_output = f"{bios_output_path}/{bios_act9_output}/bios_out_cable_1900_2022.nc"
trendy_cable_output = f"{trendy_output_path}/{trendy_act9_output}/bios_out_cable_1901_2022.nc"

bios_casa_output = f"{bios_output_path}/{bios_act9_output}/bios_out_casa_1900_2022.nc"
trendy_casa_output = f"{trendy_output_path}/{trendy_act9_output}/bios_out_casa_1901_2022.nc"

In [7]:
def preprocess_dataset(d, filter_vars):
    """Renames to longtiude/latitude, and only keeps variables listed in cable/casa"""
    p_dataset = d.drop_vars(list(filter(lambda x: x not in filter_vars, d.keys())))
    if "x" in p_dataset.dims:
        p_dataset = p_dataset.rename(name_dict={"x": "longitude", "y": "latitude"})
    return p_dataset


In [9]:
def extract_points(d, landmask):
    """Filter dataset points, given a landmask"""
    landmask_stack = landmask.stack(z=('latitude', 'longitude'))
    fil_pts = landmask_stack.where(landmask_stack["land"] != 0, drop=True)
    extractors = fil_pts.z.values
    extractors_lat = [i[0] for i in extractors]
    extractors_lon = [i[1] for i in extractors]
    return d.sel(longitude=extractors_lon, latitude=extractors_lat, method="nearest").drop_duplicates(...)

In [ ]:
def differentiate_sec_forest(d):
    """ 
    Differentiate primary and secondary forest
    Assumption: Secondary forest can only exist at index 1
    """
    new_iveg = d.sel(patch=1)["iveg"]
    new_iveg = new_iveg.where(lambda x: x != 2, -2)
    d["iveg"].loc[dict(patch=1)] = new_iveg
    return d

In [78]:
# Only used for vector-based outputs
def land_vec_to_loc(d):
    temp = d.set_index(land=["local_lat", "local_lon"])
    temp = temp.unstack("land")
    if "latitude" in d.dims:
        temp = temp.drop_vars(["latitude", "longitude"])
    temp = temp.rename(name_dict={"local_lon": "longitude", "local_lat": "latitude"})
    return temp

In [79]:
def setup_additional_vars(is_land_vec=False):
    additional_vars = ["iveg"]
    additional_vars += ["local_lat", "local_lon"] if is_land_vec else []
    return additional_vars

def run_cable_workflow(p, l, filter_vars, is_land_vec=False):
    d = xr.open_dataset(p)

    # Keep certain variables in dataset
    additional_vars = setup_additional_vars(is_land_vec)
    d = preprocess_dataset(d, filter_vars + additional_vars)

    # Select time range
    d = d.sel(time=slice("1901-01-01", "2022-01-01")) # TODO: Change to whatever has the smaller dim 

    # TRENDY merged outputs - only pick points from landmask
    if not is_land_vec:
        d = extract_points(d, l)

    # Secondary Evergreen Broadloaf classification
    d = differentiate_sec_forest(d)

    # 1D -> 2D dimensionality
    if is_land_vec:
        d = land_vec_to_loc(d)

    return d

In [ ]:
# Analysis 1: Heatmaps
cable_vars = [
"Rnet",
"Qh",
"Qle",
"Qs",
"Qsb",
"GPP",
"NPP",
"NEE",
]

In [ ]:
bios_cable_dataset = run_cable_workflow(bios_cable_output, landmask, cable_vars, is_land_vec=True)
bios_cable_dataset

<xarray.Dataset> Size: 1MB
Dimensions:    (latitude: 3, longitude: 3, time: 1452, patch: 3)
Coordinates:
  * latitude   (latitude) float32 12B -35.5 -35.45 -35.4
  * longitude  (longitude) float32 12B 149.4 149.5 149.6
  * time       (time) datetime64[ns] 12kB 1901-01-16T12:00:00 ... 2021-12-16T...
Dimensions without coordinates: patch
Data variables:
    Qle        (time, patch, latitude, longitude) float32 157kB 90.39 ... 113.4
    Qh         (time, patch, latitude, longitude) float32 157kB 94.82 ... 19.66
    Qs         (time, patch, latitude, longitude) float32 157kB 0.0 ... 5.451...
    Qsb        (time, patch, latitude, longitude) float32 157kB 0.0 ... 2.846...
    NEE        (time, patch, latitude, longitude) float32 157kB -0.08288 ... ...
    Rnet       (time, patch, latitude, longitude) float32 157kB 178.3 ... 137.2
    GPP        (time, patch, latitude, longitude) float32 157kB 5.63 ... 13.62
    NPP        (time, patch, latitude, longitude) float32 157kB 3.437 ... 9.048
    iveg       (patch, latitude, longitude) float64 216B 2.0 2.0 2.0 ... 6.0 6.0
Attributes:
    Production:        2024/05/09 at 11:56:19
    Source:            CABLE LSM output file
    CABLE_input_file:  bios
    Output_averaging:  monthly

In [82]:
trendy_cable_dataset = run_cable_workflow(trendy_cable_output, landmask, cable_vars)
trendy_cable_dataset = trendy_cable_dataset.load()
trendy_cable_dataset

<xarray.Dataset> Size: 1MB
Dimensions:    (longitude: 3, latitude: 3, patch: 3, time: 1452)
Coordinates:
  * longitude  (longitude) float32 12B 149.4 149.5 149.6
  * latitude   (latitude) float32 12B -35.5 -35.45 -35.4
  * time       (time) datetime64[ns] 12kB 1901-01-16T12:00:00 ... 2021-12-16T...
Dimensions without coordinates: patch
Data variables:
    iveg       (patch, latitude, longitude) float64 216B 2.0 2.0 2.0 ... 6.0 6.0
    Qle        (time, patch, latitude, longitude) float32 157kB 99.87 ... 92.12
    Qh         (time, patch, latitude, longitude) float32 157kB 74.78 ... 48.43
    Qs         (time, patch, latitude, longitude) float32 157kB 0.0 ... 6.17e-06
    Qsb        (time, patch, latitude, longitude) float32 157kB 0.0 ... 3.522...
    NEE        (time, patch, latitude, longitude) float32 157kB 0.5383 ... -1...
    Rnet       (time, patch, latitude, longitude) float32 157kB 173.5 ... 135.2
    GPP        (time, patch, latitude, longitude) float32 157kB 4.119 ... 4.579
    NPP        (time, patch, latitude, longitude) float32 157kB 1.98 ... 3.114
Attributes:
    Production:        2025/03/31 at 14:57:53
    Source:            CABLE LSM output file
    CABLE_input_file:  
    Output_averaging:  monthly
    history:           Mon Apr  7 15:53:55 2025: /g/data/rp23/experiments/202...

In [ ]:
casa_vars = [
"clabile",
"cplant",
"csoil",
"clitter",
]


In [ ]:
# TODO: Work on after CABLE outputs are clarified
def set_casa_index(d):
    pass
def run_casa_workflow(p, l, filter_vars, iveg, is_land_vec=False):
    d = xr.open_dataset(p)
    d["patch"] = d["land"] % 3
    d = d.set_index(land=["patch", "latitude", "longitude"])
    d = preprocess_dataset(d, casa_vars)
    d = d.unstack("land")
    d["iveg"] = iveg
    return d

# xr.open_dataset(bios_casa_output)
bios_casa_dataset = run_casa_workflow(bios_casa_output, landmask, casa_vars, bios_cable_dataset["iveg"], is_land_vec=True)
bios_casa_dataset

In [87]:
def calc_results(d):
    results = {}
    print("Calculating mean")
    results["mean"] = d.mean(dim="time").compute()
    print("Calculating min")
    results["min"] = d.min(dim="time").compute()
    print("Calculating max")
    results["max"] = d.max(dim="time").compute()
    return results

In [88]:
bios_cable_results = calc_results(bios_cable_dataset)
trendy_cable_results = calc_results(trendy_cable_dataset)

Calculating mean
Calculating min
Calculating max
Calculating mean
Calculating min
Calculating max


In [ ]:
def draw_act9_heatmap(dpv, ax, title, **kwargs):
    sns.heatmap(dpv, annot=True, ax=ax, **kwargs)
    ax.set_yticklabels(dpv.longitude.values)
    ax.set_xticklabels(dpv.latitude.values)
    ax.set_title(title)
    ax.set(xlabel='latitude', ylabel='longitude')

def draw_act_dist(dpv, ndpv, ax):
    diff_vals = abs(dpv - ndpv).values.ravel()
    sns.histplot(diff_vals, ax=ax)
    plt.axvline(diff_vals.mean(), color = 'green', linestyle = 'dashed', linewidth = 3, label='mean')
    ax.set_title(f"Spatial points differences distribution: mean={diff_vals.mean():.2f}, min={diff_vals.min():.2f}, max={diff_vals.max():.2f}")


def gen_vmin_max(dpv, ndpv):
    concacted_dataset = xr.concat([dpv, ndpv], dim="latitude")
    return concacted_dataset.min().values, concacted_dataset.max().values

def generate_heatmaps(output_dir, r, n_r, unique_veg_types=veg_type_map.keys()):
    for veg_type in unique_veg_types:
        for analysis in ["mean", "min", "max"]:
            dp = r[analysis].where(r[analysis]["iveg"] == veg_type).max(dim="patch")
            n_dp = n_r[analysis].where(n_r[analysis]["iveg"] == veg_type).max(dim="patch")
            for v in cable_vars:
                vmin, vmax = gen_vmin_max(dp[v], n_dp[v])
                fig, ax = plt.subplot_mosaic("AB;CC", figsize=(15, 10))
                title = f"Var: {v} Veg_type: {veg_type_map[veg_type]} Analysis: {analysis} "
                fig.suptitle(title)
                print(title)
                draw_act9_heatmap(dp[v], ax['A'], "BIOS", vmin=vmin, vmax=vmax)
                draw_act9_heatmap(n_dp[v], ax['B'], "TRENDY", vmin=vmin, vmax=vmax)
                draw_act_dist(dp[v], n_dp[v], ax['C'])
                plt.savefig(f"{output_dir}/{v}_{veg_type_map[veg_type]}_{analysis}.png")
                plt.close("all")

In [90]:
generate_heatmaps("outputs", bios_cable_results, trendy_cable_results, unique_veg_types=np.unique(bios_cable_dataset["iveg"]))

Var: Rnet Veg_type: peb Analysis: mean 
Var: Qh Veg_type: peb Analysis: mean 
Var: Qle Veg_type: peb Analysis: mean 
Var: Qs Veg_type: peb Analysis: mean 
Var: Qsb Veg_type: peb Analysis: mean 
Var: GPP Veg_type: peb Analysis: mean 
Var: NPP Veg_type: peb Analysis: mean 
Var: NEE Veg_type: peb Analysis: mean 
Var: Rnet Veg_type: peb Analysis: min 
Var: Qh Veg_type: peb Analysis: min 
Var: Qle Veg_type: peb Analysis: min 
Var: Qs Veg_type: peb Analysis: min 
Var: Qsb Veg_type: peb Analysis: min 
Var: GPP Veg_type: peb Analysis: min 
Var: NPP Veg_type: peb Analysis: min 
Var: NEE Veg_type: peb Analysis: min 
Var: Rnet Veg_type: peb Analysis: max 
Var: Qh Veg_type: peb Analysis: max 
Var: Qle Veg_type: peb Analysis: max 
Var: Qs Veg_type: peb Analysis: max 
Var: Qsb Veg_type: peb Analysis: max 
Var: GPP Veg_type: peb Analysis: max 
Var: NPP Veg_type: peb Analysis: max 
Var: NEE Veg_type: peb Analysis: max 
Var: Rnet Veg_type: seb Analysis: mean 
Var: Qh Veg_type: seb Analysis: mean 
Var: 

In [ ]:
# Analysis 2
cable_vars_timeseries = [
    "GPP",
    "Qh",
    "Qle",
    "TVeg"
]

In [31]:
bios_cable_dataset = run_cable_workflow(bios_cable_output, landmask, cable_vars_timeseries, is_land_vec=True)
bios_cable_dataset

<xarray.Dataset> Size: 639kB
Dimensions:    (latitude: 3, longitude: 3, time: 1452, patch: 3)
Coordinates:
  * latitude   (latitude) float32 12B -35.5 -35.45 -35.4
  * longitude  (longitude) float32 12B 149.4 149.5 149.6
  * time       (time) datetime64[ns] 12kB 1901-01-16T12:00:00 ... 2021-12-16T...
Dimensions without coordinates: patch
Data variables:
    Qle        (time, patch, latitude, longitude) float32 157kB 93.37 ... 113.6
    Qh         (time, patch, latitude, longitude) float32 157kB 98.28 ... 25.05
    TVeg       (time, patch, latitude, longitude) float32 157kB 3.022e-05 ......
    GPP        (time, patch, latitude, longitude) float32 157kB 5.676 ... 14.09
    iveg       (patch, latitude, longitude) float64 216B 2.0 2.0 2.0 ... 6.0 6.0
Attributes:
    Production:        2024/05/09 at 11:56:19
    Source:            CABLE LSM output file
    CABLE_input_file:  bios
    Output_averaging:  monthly

In [101]:
trendy_cable_dataset = run_cable_workflow(trendy_cable_output, landmask, cable_vars_timeseries)
trendy_cable_dataset = trendy_cable_dataset.load()

<xarray.Dataset> Size: 40GB
Dimensions:    (longitude: 841, latitude: 681, patch: 3, time: 1464)
Coordinates:
  * longitude  (longitude) float32 3kB 112.0 112.1 112.1 ... 153.9 153.9 154.0
  * latitude   (latitude) float32 3kB -44.0 -43.95 -43.9 ... -10.1 -10.05 -10.0
  * time       (time) datetime64[ns] 12kB 1901-01-16T12:00:00 ... 2022-12-16T...
Dimensions without coordinates: patch
Data variables:
    iveg       (patch, latitude, longitude) float64 14MB ...
    Qle        (time, patch, latitude, longitude) float32 10GB ...
    Qh         (time, patch, latitude, longitude) float32 10GB ...
    TVeg       (time, patch, latitude, longitude) float32 10GB ...
    GPP        (time, patch, latitude, longitude) float32 10GB ...
Attributes:
    Production:        2025/03/31 at 14:57:53
    Source:            CABLE LSM output file
    CABLE_input_file:  
    Output_averaging:  monthly
    history:           Mon Apr  7 15:53:55 2025: /g/data/rp23/experiments/202...


In [33]:
def draw_act9_timeseries(dpv, ax, title, **kwargs):
    pass

def generate_timeseries(output_dir, d, n_d, var_list, unique_veg_types=veg_type_map.keys()):
    for veg_type in unique_veg_types:

        for lat in d["latitude"]:
            for lon in d["longitude"]:
                dp = d.sel(latitude=lat, longitude=lon)
                dp = dp.where(dp["iveg"] == veg_type).max(dim="patch") 
                n_dp = n_d.sel(latitude=lat, longitude=lon)
                n_dp = n_dp.where(n_dp["iveg"] == veg_type).max(dim="patch") 
                fig, ax = plt.subplots(math.ceil(len(var_list) / 2), 2, figsize=(20, 10))
                sup_title = f"Veg_type: {veg_type_map[veg_type]} lon: {lon.values} lat: {lat.values}"
                fig.suptitle(sup_title)

                # Used later for common label
                handles = None
                labels = None

                for i, v in enumerate(var_list):
                    print(f"{sup_title} Var: {v}")
                    # NOTE: Different day?/hour after around 1935 but same month for some values 
                    combined_plot = xr.concat([dp[v], n_dp[v]], dim=pd.Index(["BIOS", "TRENDY"]), join="override")
                    # Plot on the correct axis
                    cur_ax = ax[i // 2, i % 2]
                    # print(dp[v].time.values)
                    # print(n_dp[v].time.values)
                    # print(combined_plot.time.values)
                    combined_plot.plot.line(x="time", ax=cur_ax, alpha=0.8)
                    # Remove Title
                    cur_ax.set_title("")
                    # Remove legend (common for all)
                    handles, labels = cur_ax.get_legend_handles_labels()
                    # cur_ax.get_legend().remove()

                fig.legend(handles, labels, loc='upper center')
                plt.tight_layout()
                plt.savefig(f"{output_dir}/lat-{lat.values:.2f}_lon-{lon.values:.2f}_{veg_type_map[veg_type]}.png")
                plt.close("all")

In [34]:
generate_timeseries("output_timeseries", bios_cable_dataset, trendy_cable_dataset, cable_vars_timeseries, unique_veg_types=[2, 6])

Veg_type: seb lon: 149.4499969482422 lat: -35.5 Var: GPP
Veg_type: seb lon: 149.4499969482422 lat: -35.5 Var: Qh
Veg_type: seb lon: 149.4499969482422 lat: -35.5 Var: Qle
Veg_type: seb lon: 149.4499969482422 lat: -35.5 Var: TVeg
Veg_type: seb lon: 149.5 lat: -35.5 Var: GPP
Veg_type: seb lon: 149.5 lat: -35.5 Var: Qh
Veg_type: seb lon: 149.5 lat: -35.5 Var: Qle
Veg_type: seb lon: 149.5 lat: -35.5 Var: TVeg
Veg_type: seb lon: 149.5500030517578 lat: -35.5 Var: GPP
Veg_type: seb lon: 149.5500030517578 lat: -35.5 Var: Qh
Veg_type: seb lon: 149.5500030517578 lat: -35.5 Var: Qle
Veg_type: seb lon: 149.5500030517578 lat: -35.5 Var: TVeg
Veg_type: seb lon: 149.4499969482422 lat: -35.45000076293945 Var: GPP
Veg_type: seb lon: 149.4499969482422 lat: -35.45000076293945 Var: Qh
Veg_type: seb lon: 149.4499969482422 lat: -35.45000076293945 Var: Qle
Veg_type: seb lon: 149.4499969482422 lat: -35.45000076293945 Var: TVeg
Veg_type: seb lon: 149.5 lat: -35.45000076293945 Var: GPP
Veg_type: seb lon: 149.5 l

In [93]:
cable_vars_moist = [
    "SoilMoist",
    "swilt",
    "patchfrac"
]

def aggregate_patchfrac(d, var: str):
    return (d[var] * d["patchfrac"]).sum(dim="patch")

In [94]:
bios_cable_dataset = run_cable_workflow(bios_cable_output, landmask, cable_vars_moist, is_land_vec=True)
bios_cable_dataset["SoilMoist"] = aggregate_patchfrac(bios_cable_dataset, "SoilMoist")
bios_cable_dataset["swilt"] = aggregate_patchfrac(bios_cable_dataset, "swilt")
bios_cable_dataset

<xarray.Dataset> Size: 326kB
Dimensions:    (latitude: 3, longitude: 3, time: 1452, soil: 6, patch: 3)
Coordinates:
  * latitude   (latitude) float32 12B -35.5 -35.45 -35.4
  * longitude  (longitude) float32 12B 149.4 149.5 149.6
  * time       (time) datetime64[ns] 12kB 1901-01-16T12:00:00 ... 2021-12-16T...
Dimensions without coordinates: soil, patch
Data variables:
    SoilMoist  (time, soil, latitude, longitude) float32 314kB 0.1481 ... 0.2009
    iveg       (patch, latitude, longitude) float64 216B 2.0 2.0 2.0 ... 6.0 6.0
    patchfrac  (patch, latitude, longitude) float32 108B 0.4767 0.4767 ... 0.0
    swilt      (latitude, longitude) float32 36B 0.136 0.135 ... 0.135 0.136
Attributes:
    Production:        2024/05/09 at 11:56:19
    Source:            CABLE LSM output file
    CABLE_input_file:  bios
    Output_averaging:  monthly

In [95]:
trendy_cable_dataset = run_cable_workflow(trendy_cable_output, landmask, cable_vars_moist)
trendy_cable_dataset["swilt"] = aggregate_patchfrac(trendy_cable_dataset, "swilt")
trendy_cable_dataset = trendy_cable_dataset.load()

In [ ]:

def generate_timeseries_moist(output_dir, d, n_d):
    for lat in d["latitude"]:
        for lon in d["longitude"]:
            # Select point
            dp = d.sel(latitude=lat, longitude=lon)
            n_dp = n_d.sel(latitude=lat, longitude=lon)
            # Create subplot
            fig, ax = plt.subplots(math.ceil(d["soil"].size / 2), 2, figsize=(20, 10))
            sup_title = f"lon: {lon.values:.2f} lat: {lat.values:.2f}"
            fig.suptitle(sup_title)
            # Used later for common label
            handles = None
            labels = None
            for i in range(d["soil"].size):
                vmin, vmax = gen_vmin_max(dp["SoilMoist"], n_dp["SoilMoist"])
                print(f"{sup_title} Soil: {i}")
                soilmoist_dp = dp["SoilMoist"].sel(soil=i)
                soilmoist_ndp = n_dp["SoilMoist"].sel(soil=i)
                cur_ax = ax[i // 2, i % 2]
                combined_plot = xr.concat([soilmoist_dp, soilmoist_ndp], join="override")
                # Plot on the relevant axis
                combined_plot.plot.line(x="time", ax=cur_ax, alpha=0.8, ylim=(vmin, vmax), add_legend=False)
                cur_ax.legend(labels=["BIOS", "TRENDY"], loc="upper right")
                # Horizontal lines for swilt
                cur_ax.axhline(dp["swilt"], color = 'green', linestyle = 'dashed', linewidth = 3, label='swilt_BIOS', alpha=1)
                cur_ax.axhline(n_dp["swilt"], color = 'red', linestyle = 'dashed', linewidth = 3, label='swilt_TRENDY', alpha=1)
                # Remove Title
                cur_ax.set_title("")
                # Remove legend (common for all)
                handles, labels = cur_ax.get_legend_handles_labels()
                print(handles, labels)
                # cur_ax.get_legend().remove()
            fig.legend(handles, labels, loc='upper right')
            plt.tight_layout()
            plt.savefig(f"{output_dir}/lat-{lat.values:.2f}_lon-{lon.values:.2f}.png")
            plt.close("all")

In [109]:
generate_timeseries_moist("output_moist", bios_cable_dataset, trendy_cable_dataset)

lon: 149.45 lat: -35.50 Soil: 0
[<matplotlib.lines.Line2D object at 0x7f73a2dcc2b0>, <matplotlib.lines.Line2D object at 0x7f73a2dcea70>] ['swilt_BIOS', 'swilt_TRENDY']
lon: 149.45 lat: -35.50 Soil: 1
[<matplotlib.lines.Line2D object at 0x7f73a26980a0>, <matplotlib.lines.Line2D object at 0x7f73a3936aa0>] ['swilt_BIOS', 'swilt_TRENDY']
lon: 149.45 lat: -35.50 Soil: 2
[<matplotlib.lines.Line2D object at 0x7f73a2dcc730>, <matplotlib.lines.Line2D object at 0x7f73a39356f0>] ['swilt_BIOS', 'swilt_TRENDY']
lon: 149.45 lat: -35.50 Soil: 3
[<matplotlib.lines.Line2D object at 0x7f73a3937ee0>, <matplotlib.lines.Line2D object at 0x7f73a365f970>] ['swilt_BIOS', 'swilt_TRENDY']
lon: 149.45 lat: -35.50 Soil: 4
[<matplotlib.lines.Line2D object at 0x7f73a37e7400>, <matplotlib.lines.Line2D object at 0x7f73a365e9e0>] ['swilt_BIOS', 'swilt_TRENDY']
lon: 149.45 lat: -35.50 Soil: 5
[<matplotlib.lines.Line2D object at 0x7f73a365e500>, <matplotlib.lines.Line2D object at 0x7f73a5af6470>] ['swilt_BIOS', 'swilt_T

In [102]:
time_diffs = bios_cable_dataset.time.to_index() - trendy_cable_dataset.time.to_index()
time_diffs.unique()

TimedeltaIndex([  '0 days 00:00:00', '-1 days +23:58:56',   '0 days 00:01:04',
                  '0 days 00:02:08', '-1 days +23:57:52'],
               dtype='timedelta64[ns]', name='time', freq=None)